# Phase 1: Teacher-Student Framework
In this phase, we initialize a pretrained GPT-2 Small (124M) model. We duplicate it into a Teacher and a Student. We then feed a 4-shot prompt to the Teacher and a 0-shot prompt to the Student to verify that they produce different hidden states for the final token (which is used to predict the next word).

In [12]:
!pip install transformers torch

In [13]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and model
model_name = "gpt2"  # 124M parameter model
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Initialize Teacher and Student models
teacher_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
student_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# Freeze both models (as per Phase 1 description)
for param in teacher_model.parameters():
    param.requires_grad = False
    
for param in student_model.parameters():
    param.requires_grad = False

teacher_model.eval()
student_model.eval()
print("Models loaded and frozen.")

Using device: cpu
Models loaded and frozen.


In [14]:
# Define the prompts
# We use a sentiment classification task to ensure the teacher's context meaningfully affects the prediction.
few_shot_prompt = """Review: I loved this movie, it was fantastic!
Sentiment: Positive

Review: This was the worst experience of my life.
Sentiment: Negative

Review: The food was okay, but the service was terrible.
Sentiment: Negative

Review: Absolutely brilliant performance by the actors.
Sentiment: Positive

Review: The product broke after two days of use.
Sentiment:"""

zero_shot_prompt = """Review: The product broke after two days of use.
Sentiment:"""

print("Teacher Prompt (4-shot):")
print("-" * 30)
print(few_shot_prompt)
print("-" * 30)
print("\nStudent Prompt (0-shot):")
print("-" * 30)
print(zero_shot_prompt)
print("-" * 30)

Teacher Prompt (4-shot):
------------------------------
Review: I loved this movie, it was fantastic!
Sentiment: Positive

Review: This was the worst experience of my life.
Sentiment: Negative

Review: The food was okay, but the service was terrible.
Sentiment: Negative

Review: Absolutely brilliant performance by the actors.
Sentiment: Positive

Review: The product broke after two days of use.
Sentiment:
------------------------------

Student Prompt (0-shot):
------------------------------
Review: The product broke after two days of use.
Sentiment:
------------------------------


In [15]:
# Tokenize prompts
teacher_inputs = tokenizer(few_shot_prompt, return_tensors="pt").to(device)
student_inputs = tokenizer(zero_shot_prompt, return_tensors="pt").to(device)

print(f"Teacher sequence length: {teacher_inputs['input_ids'].shape[1]}")
print(f"Student sequence length: {student_inputs['input_ids'].shape[1]}")

# Forward pass, outputting hidden states
with torch.no_grad():
    teacher_outputs = teacher_model(**teacher_inputs, output_hidden_states=True)
    student_outputs = student_model(**student_inputs, output_hidden_states=True)

# The hidden states is a tuple of (layers + 1) tensors.
# Let's extract the last layer's hidden states.
teacher_last_hidden_state = teacher_outputs.hidden_states[-1] # Shape: (batch, seq_len_t, hidden_dim)
student_last_hidden_state = student_outputs.hidden_states[-1] # Shape: (batch, seq_len_s, hidden_dim)

# We are interested in the hidden state of the very last token, which predicts the answer.
teacher_final_token_hs = teacher_last_hidden_state[:, -1, :] # Shape: (batch, hidden_dim)
student_final_token_hs = student_last_hidden_state[:, -1, :] # Shape: (batch, hidden_dim)

print(f"Teacher final token hidden state shape: {teacher_final_token_hs.shape}")
print(f"Student final token hidden state shape: {student_final_token_hs.shape}")

Teacher sequence length: 87
Student sequence length: 15
Teacher final token hidden state shape: torch.Size([1, 768])
Student final token hidden state shape: torch.Size([1, 768])


In [16]:
import torch.nn.functional as F

# Compute differences between the final hidden states
mse_diff = F.mse_loss(teacher_final_token_hs, student_final_token_hs)
cos_sim = F.cosine_similarity(teacher_final_token_hs, student_final_token_hs)

print(f"Mean Squared Error between hidden states: {mse_diff.item():.6f}")
print(f"Cosine Similarity between hidden states: {cos_sim.item():.4f}")

if not torch.allclose(teacher_final_token_hs, student_final_token_hs, atol=1e-5):
    print("\n✅ Verification Successful: The teacher and student produce different hidden states for the final token.")
else:
    print("\n❌ Verification Failed: The hidden states are identical.")

Mean Squared Error between hidden states: 84.421486
Cosine Similarity between hidden states: 0.6201

✅ Verification Successful: The teacher and student produce different hidden states for the final token.
